In [15]:
import pandas as pd  # импорты
import numpy as np  # код по стандартам pep8
from sklearn.metrics import f1_score, classification_report  # для f1
import torch
import torch.nn as nn
import torch.optim as optim  # оптимизаторы
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models  # для работы с изображениями
from PIL import Image  # для загрузки изображений
from sklearn.model_selection import train_test_split
from tqdm import tqdm  # прогресс бар
import os  # нужно для того если я прерву обучение чтобы оно не начиналось с нуля

train_df = pd.read_csv('train.csv')  # решил поместить загрузку данных сюда
test_df = pd.read_csv('test.csv')


In [16]:
class PriceDataset(Dataset):
    def __init__(self, df, img_dir='images', transform=None, is_train=True):
        self.df = df  # инициализация
        self.img_dir = img_dir
        self.transform = transform
        self.is_train = is_train

    def __len__(self):  # тут все понятно
        return len(self.df)

    def __getitem__(self, idx):  # получение элемента
        img_id = str(self.df.iloc[idx]['id'])
        img_path = os.path.join(self.img_dir, f"{img_id}.webp")  # загрузка изображения
        image = Image.open(img_path).convert('RGB')  # преобразование в RGB (на всякий случай)
        if self.transform:
            image = self.transform(image)  # применение трансформаций
        result = {
            'image': image,
            'img_id': img_id
        }
        if self.is_train:
            result['label'] = torch.tensor(self.df.iloc[idx]['class_id'], dtype=torch.long)  # метка класса
        return result

In [17]:
class PriceModel(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.backbone = models.efficientnet_b3(pretrained=True)  # легкая модель
        num_features = self.backbone.classifier[1].in_features  # получаем размер признаков

        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=0.9, inplace=True),  # вероятность отключения нейронов т.к. датасет очень маленький
            nn.Linear(num_features, num_classes)  # напрямую к выходу
        )

    def forward(self, x):
        return self.backbone(x)

In [18]:
transform = transforms.Compose([  # чтобы все данные были одинаковыми
    transforms.Resize((456, 456)),  # все аугментации которые помню
    transforms.RandomRotation(degrees=15),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),  # преобразование в тензор
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # нормализация
])
train_data, val_data = train_test_split(train_df, test_size=0.2,
                                        stratify=train_df['class_id'],
                                        random_state=42)  # 80% на обучение остальное на валидацию
train_dataset = PriceDataset(train_data, 'images', transform, True)  # для тренировки
val_dataset = PriceDataset(val_data, 'images', transform, True)  # для валидации
test_dataset = PriceDataset(test_df, 'images', transform, False)  # для тестирования
train_loader = DataLoader(train_dataset, batch_size=8,
                          shuffle=True)  # батч такой из-за ограничений vram + перемешивание результатов
val_loader = DataLoader(val_dataset, batch_size=24, shuffle=False)  # тут не принципиален батч сайз(т.к. нет градиентов)
test_loader = DataLoader(test_dataset, batch_size=24, shuffle=False)  # нет перемешивания результатов

In [ ]:
device = torch.device('cuda')  # обучал на видеокарте
model = PriceModel().to(device)
if os.path.exists('best_model3.pth'):  # если есть модель то загружаю
    model.load_state_dict(torch.load('best_model3.pth'))
optimizer = optim.Adam(model.parameters(), lr=4e-5)  # ставил руками
criterion = nn.CrossEntropyLoss()  # функция потерь
best_f1 = 0  # сохранение по f1
for epoch in range(20):  # не принципиально я часто останавливал обучение
    model.train()
    train_loss = 0  # обучение
    for batch in tqdm(train_loader, leave=False):  # прогресс бар для красоты
        images = batch['image'].to(device)  # изображения
        labels = batch['label'].to(device)  # метки классов
        optimizer.zero_grad()
        preds = model(images)  # предсказания модели
        loss = criterion(preds, labels)  # считаем loss
        loss.backward()
        optimizer.step()
        train_loss += loss.item()  # train loss
    model.eval()
    val_loss = 0
    all_predictions = []  # предсказания
    all_true_labels = []  # истинные метки
    with torch.no_grad():  # валидация
        for batch in val_loader:
            images = batch['image'].to(device)
            labels = batch['label'].to(device)
            preds = model(images)
            loss = criterion(preds, labels)
            val_loss += loss.item()  # val loss
            all_predictions.extend(torch.max(preds, 1)[1].cpu().numpy())
            all_true_labels.extend(labels.cpu().numpy())
    f1 = f1_score(all_true_labels, all_predictions, average='macro')  # единственная метрика которую я умею считать
    avg_train_loss = train_loss / len(train_loader)  # средние train и val loss
    avg_val_loss = val_loss / len(val_loader)
    print(
        f"эпоха {epoch + 1} train Loss {avg_train_loss:.4f}, val Loss {avg_val_loss:.4f}, f1 {f1:.4f}")  # округлеие до 4 знаков
    if f1 > best_f1:  # сохраняю по лучшему F1
        best_f1 = f1
        torch.save(model.state_dict(), 'best_model3.pth')
        print("сохранена лучшая модель")

In [ ]:
model.eval()
preds_list = []
with torch.no_grad():
    for batch in test_loader:
        images = batch['image'].to(device)
        outputs = model(images)
        preds_list.extend(torch.argmax(outputs, 1).cpu().numpy())
pd.DataFrame({'id': test_df['id'], 'y_pred': preds_list}).to_csv('submission.csv', index=False)
print("все")